In [ ]:
!pip install suntime

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time

from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess

from sklearn.utils.validation import _check_fit_params
from sklearn.base import is_classifier
from joblib import Parallel, delayed
from sklearn.multioutput import _fit_estimator


from sklearn.model_selection import KFold
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error

from xgboost import XGBRegressor
from catboost import CatBoostRegressor

from suntime import Sun
import pytz


import warnings
warnings.filterwarnings("ignore")

In [ ]:

#Functions
def encode(data, col, max_val):
    data[col + '_sin'] = np.sin(2 * np.pi * data[col]/max_val)
    data[col + '_cos'] = np.cos(2 * np.pi * data[col]/max_val)
    return data

def creat_time_features(df):
    """
    Creates time series features from datetime index
    """
    df = df.copy()
    df['year'] = df.DateTime.dt.year
    df['dayofweek'] = df.DateTime.dt.dayofweek
    df['weekofyear'] = df.DateTime.dt.week
    df['quarter'] = df.DateTime.dt.quarter
    df['month'] = df.DateTime.dt.month
    df['dayofyear'] = df.DateTime.dt.dayofyear
    df['day'] = df.DateTime.dt.day
    df['hour'] = df.DateTime.dt.hour
    df = encode(df, 'dayofyear', 365)
    df = encode(df, 'month', 12)
    return df

def get_sun_angle(df):
    df_c = df.copy()
    df_c['hour'] = df_c.DateTime.dt.hour
    df_c['date'] = df_c.DateTime.dt.date
    sunrise_set_hours = df_c[df_c.is_date == True].groupby('date').agg({'hour':['min', 'max']}).reset_index()
    sunrise_set_hours.columns = ['date', 'sunrise', 'sunset']
    sunrise_set_hours = pd.merge(df_c, sunrise_set_hours, how='left', on='date')
    
    def get_angle(hour, start, end):
        if hour>= start and hour <= end:
            return np.sin(np.pi*(hour - start + 1)/(end - start + 2))
        else:
            return 0
        
    sunrise_set_hours['sun_angle'] = sunrise_set_hours.apply(lambda x: get_angle(x.hour, x.sunrise, x.sunset), axis=1)
    
    df_c = pd.concat([df_c,sunrise_set_hours.sun_angle], axis=1)
    df_c.drop(['date', 'hour'], axis=1, inplace=True)
    return df_c

def temp_corrections(df):
    df = df.copy()
    df.WWCode.fillna(0, inplace=True)
    df.WWCode = df.WWCode.replace([84], 83)
    df.DateTime = pd.to_datetime(df.DateTime)
    df.AirTemperature = df.AirTemperature.apply(lambda x: float(str(x).replace(',', '.')))
    df.ComfortTemperature = df.ComfortTemperature.apply(lambda x: float(str(x).replace(',', '.')))
    df.RelativeHumidity = df.RelativeHumidity.apply(lambda x: float(str(x).replace(',', '.')))
    df.WindSpeed = df.WindSpeed.apply(lambda x: float(str(x).replace(',', '.')))
    df.EffectiveCloudCover = df.EffectiveCloudCover.apply(lambda x: float(str(x).replace(',', '.')))
    df = encode(df, 'WindDirection', 360)
    df.dropna(inplace=True)
    
    fourier = CalendarFourier(freq='A', order=10)
    dp = DeterministicProcess(index=df.DateTime,
                              constant=True,
                              order=1,
                              seasonal=False,
                              additional_terms=[fourier],
                              drop=True)
    
    additional_datetime_features = dp.in_sample()

    df = pd.concat([df, additional_datetime_features.reset_index(drop=True)], axis=1)
    
    coordinates = [40.239, 33.029] # Ankara
    
    df["Date"] = df["DateTime"].apply(pd.to_datetime).dt.date.apply(str)
    df["Hour"] = df["DateTime"].apply(pd.to_datetime).dt.hour
    
    sun = Sun(coordinates[0], coordinates[1])
    tz =pytz.timezone('Europe/Istanbul')
    df["is_date"]= df[["Date","Hour"]].apply(lambda x : sun.get_local_sunrise_time(pd.to_datetime(x["Date"]).date(),
                                                                               local_time_zone=tz).hour <= x["Hour"] <= sun.get_local_sunset_time(pd.to_datetime(x["Date"]).date(),
                                                                                                                                                  local_time_zone=tz).hour ,axis=1)
    df = df.drop(['Date', 'Hour'], axis=1)
    
    df = get_sun_angle(df)
    return df

def gen_corrections(df):
    df = df.copy()
    df.dropna(inplace=True)
    df.DateTime = pd.to_datetime(df.DateTime)
    df.Generation = df.Generation.apply(lambda x: float(str(x).replace(',', '.')))
    return df


def get_model_data(df, window_start, target=False):
    if target == False:
        if 'Generation' in df.columns:
            X = df[df.DateTime >= window_start].drop('Generation', axis=1) 
        else:
            X = df
        X['date'] = X.DateTime.dt.date
        X['DateTime'] = X.DateTime.dt.hour
        X = X.set_index(['date']).sort_index()
        X = X.groupby(['date', 'DateTime']).sum().unstack('DateTime')
        return X
    else:
        y = df[df.DateTime >= window_start][['DateTime', 'Generation']]
        y['date'] = y.DateTime.dt.date
        y['DateTime'] = y.DateTime.dt.hour
        y = y.set_index(['date']).sort_index()
        y = y.groupby(['date', 'DateTime']).agg({'Generation':'sum'}).unstack('DateTime')
        return y
    

#Data Loading
gen_df = pd.read_csv('../input/enerjisa-enerji-veri-maratonu/generation.csv', delimiter=';')
temp_df = pd.read_csv('../input/enerjisa-enerji-veri-maratonu/temperature.csv', delimiter=';')
test_df = pd.read_csv('../input/enerjisa-enerji-veri-maratonu/sample_submission.csv')

test_df = pd.concat([gen_df, test_df])
test_df = gen_corrections(test_df)

#Pre-processing
gen_df = gen_corrections(gen_df)
temp_df = temp_corrections(temp_df)

df = pd.merge(gen_df, temp_df, how='left', on='DateTime')
df = creat_time_features(df)

all_gen_df = pd.concat([gen_df,test_df])
all_gen_df.Generation = all_gen_df.apply(lambda x: np.nan if x.DateTime >= pd.to_datetime('2021-12-01 00:00:00') else x.Generation, axis=1)
all_gen_df.drop_duplicates(inplace=True)
df_c = pd.merge(all_gen_df, temp_df, how='left', on='DateTime')
df_c = creat_time_features(df_c)
df_c = df_c[['DateTime', 'Generation', 'sun_angle', 'WindSpeed', 'WWCode', 'EffectiveCloudCover', 'RelativeHumidity', 'ComfortTemperature', 'AirTemperature',
       'dayofyear_cos']]


#Rollings
SHIFT_DAYS = [31, 31*2, 31*3, 365, 365*2]

for SHIFT_DAY in SHIFT_DAYS:
    for i in [2,3,4,5,6,7]:
        df_c['shiftday_' +str(SHIFT_DAY) + '_rolling_mean_'+str(i)] = df_c.Generation.transform(lambda x: x.shift(SHIFT_DAY*24).rolling(i).mean(skipna=True).astype(np.float16))
        df_c['shiftday_' +str(SHIFT_DAY) + '_rolling_std_'+str(i)]  = df_c.Generation.transform(lambda x: x.shift(SHIFT_DAY*24).rolling(i).std(skipna=True).astype(np.float16))

#Mean/STD Encoding
cols = ['sun_angle', 'WindSpeed', 'WWCode', 'EffectiveCloudCover', 'RelativeHumidity', 'ComfortTemperature', 'AirTemperature',
       'dayofyear_cos']

for col in cols:
    col_name = '_' + col + '_'
    df_c['enc'+col_name+'mean'] = df_c.groupby(col)['Generation'].transform('mean').astype(np.float16)
    df_c['enc'+col_name+'std'] = df_c.groupby(col)['Generation'].transform('std').astype(np.float16)
    
df_c.drop(['Generation', 'sun_angle', 'WindSpeed', 'WWCode',
       'EffectiveCloudCover', 'RelativeHumidity', 'ComfortTemperature',
       'AirTemperature', 'dayofyear_cos'], axis=1, inplace=True)

df = pd.merge(df, df_c[df_c.DateTime < pd.to_datetime('2021-12-01 00:00:00')], how='left', on='DateTime')


#Constants
SEED = 42
test_window_start = pd.to_datetime('2021-12-01')
train_winow_start = pd.to_datetime('2019-01-01')

#Test Data Prep.
test_df = test_df[test_df.DateTime >= test_window_start].drop('Generation', axis=1)
X_test = temp_df[temp_df.DateTime >= test_window_start]
X_test = pd.merge(test_df, X_test, how='left', on='DateTime')
X_test = pd.merge(X_test, df_c[df_c.DateTime >= pd.to_datetime('2021-12-01 00:00:00')], how='left', on='DateTime')
X_test = creat_time_features(X_test)

X_test = get_model_data(X_test, test_window_start, target=False)

#Train Data Prep.
X = get_model_data(df, train_winow_start, target=False) #Feature
y = get_model_data(df, train_winow_start, target=True) #Target

In [ ]:
    class MyMultiOutputRegressor(MultiOutputRegressor):
        """
        sklearn's multioutputregressor does not support XGB validation on multioutput regression.
        """
        
        def fit(self, X_train, y_train, X_val, y_val, sample_weight=None, **fit_params):
            if not hasattr(self.estimator, "fit"):
                raise ValueError("The base estimator should implement"
                                 " a fit method")
    
            X, y = self._validate_data(X_train, y_train,
                                       force_all_finite=False,
                                       multi_output=True, accept_sparse=True)
    
            if is_classifier(self):
                check_classification_targets(y_train)
    
            if y.ndim == 1:
                raise ValueError("y must have at least two dimensions for "
                                 "multi-output regression but has only one.")
    
            if (sample_weight is not None and
                    not has_fit_parameter(self.estimator, 'sample_weight')):
                raise ValueError("Underlying estimator does not support"
                                 " sample weights.")
    
            fit_params_validated = _check_fit_params(X_train, fit_params)
            self.estimators_ = Parallel(n_jobs=self.n_jobs)(
                delayed(_fit_estimator)(
                    self.estimator, X_train, y_train[:, i], sample_weight,
                    **fit_params_validated, eval_set=[(X_train, y_train[:, i]), (X_val, y_val[:, i])])
                for i in range(y_train.shape[1]))
            return

In [ ]:
#10-Fold LR
kf = KFold(n_splits=10, shuffle=True, random_state=1)
score_list = []
oof = np.zeros((y.shape))
test_preds = []
fold = 1
oof_cat_preds = np.zeros((y.shape))
oof_xgb_preds = np.zeros((y.shape))

catboost_params = {'learning_rate': 0.03,
                    'loss_function': 'MultiRMSE', 
                   'eval_metric': 'MultiRMSE', 
                   'task_type': 'CPU',
                   'boosting_type': 'Plain',
                   'bootstrap_type': 'Bayesian',
                   'iterations': 100000
                   }

xgb_params = {
    'learning_rate': 0.017659558136286068,
    'reg_lambda': 0.007463050468829971,
    'reg_alpha': 0.0713858474799285,
    'subsample': 0.5005741515919144,
    'colsample_bytree': 0.9831642464291973,
    'max_depth': 6,
    'n_estimators': 100000,
    'n_jobs':-1,
    'tree_method': "gpu_hist",
    'predictor': "gpu_predictor"
}

xgb_fit_params = dict(
     early_stopping_rounds=100,
     verbose=False
     )

for train_index, val_index in kf.split(X):
    
    X_train, X_val = X.iloc[train_index], X.iloc[val_index]
    y_train, y_val = y.iloc[train_index], y.iloc[val_index]
    
    X_train, X_val = X_train.to_numpy(), X_val.to_numpy()
    y_train, y_val = y_train.to_numpy(), y_val.to_numpy()
    
    #Model
    xgb = MyMultiOutputRegressor(XGBRegressor(**xgb_params))
    cat = CatBoostRegressor(**catboost_params)

    #Fit
    xgb.fit(X_train, y_train, X_val, y_val, **xgb_fit_params)
    cat.fit(X_train, y_train,
            eval_set = (X_val, y_val),
            use_best_model = True,
            early_stopping_rounds = 100,
            verbose_eval=0,
           )
    
    
    #Val Pred
    xgb_preds = xgb.predict(X_val)
    cat_preds = cat.predict(X_val)
    avg_preds = np.average([xgb_preds, cat_preds], axis=0)
    
    #Residual Model
    cat_y_pred = cat.predict(X_train)
    xgb_y_pred = xgb.predict(X_train)
    avg_y_pred = np.average([cat_y_pred, xgb_y_pred], axis=0)
    y_res = y_train - avg_y_pred
    y_val_res = y_val - avg_preds
    
    xgb_2 = MyMultiOutputRegressor(XGBRegressor(**xgb_params))
    cat_2 = CatBoostRegressor(**catboost_params)
    cat_2.fit(X_train, y_res,
              eval_set = (X_val, y_val_res),
              use_best_model = True,
              early_stopping_rounds = 100,
              verbose_eval=0)
    xgb_2.fit(X_train, y_res, X_val, y_val_res, **xgb_fit_params)
    
    #Residual Model - 2
    cat_y_pred = cat.predict(X_train)
    cat_y_pred_2 = cat_2.predict(X_train)
    xgb_y_pred = xgb.predict(X_train)
    xgb_y_pred_2 = xgb_2.predict(X_train)
    cat_y_pred = cat_y_pred + cat_y_pred_2
    xgb_y_pred = xgb_y_pred + xgb_y_pred_2
    avg_y_pred = np.average([cat_y_pred, xgb_y_pred], axis=0)
    y_res_2 = y_train - avg_y_pred
    y_val_res_2 = y_val - avg_preds
    
    xgb_3 = MyMultiOutputRegressor(XGBRegressor(**xgb_params))
    cat_3 = CatBoostRegressor(**catboost_params)
    cat_3.fit(X_train, y_res_2,
              eval_set = (X_val, y_val_res_2),
              use_best_model = True,
              early_stopping_rounds = 100,
              verbose_eval=0)
    xgb_3.fit(X_train, y_res_2, X_val, y_val_res_2, **xgb_fit_params)
    
    cat_preds = cat.predict(X_val) + cat_2.predict(X_val) + cat_3.predict(X_val)
    xgb_preds = xgb.predict(X_val) + xgb_2.predict(X_val) + xgb_3.predict(X_val)
    avg_preds = np.average([cat_preds, xgb_preds], axis=0)
    avg_preds = np.where(avg_preds < 0, 0, avg_preds)
    
    #Test Pred
    xgb_test_pred = xgb.predict(X_test) + xgb_2.predict(X_test) + xgb_3.predict(X_test)
    cat_test_pred = cat.predict(X_test) + cat_2.predict(X_test) + cat_3.predict(X_test)
    avg_test_preds = np.average([xgb_test_pred, cat_test_pred], axis=0)
    avg_test_preds = np.where(avg_test_preds < 0, 0, avg_test_preds)
    
    #Records
    oof[val_index] = avg_preds
    oof_xgb_preds[val_index] = xgb_preds
    oof_cat_preds[val_index] = cat_preds
    test_preds.append(avg_test_preds)
    
    #Metrics
    rmse_score = np.sqrt(mean_squared_error(y_val, avg_preds))
    score_list.append(rmse_score)
    print("Fold {0}, RMSE Score: {1}".format(fold , rmse_score))
    print("------------")
    fold += 1
    
print("------------")
print("Avg RMSE Score: ", np.average(score_list))

In [ ]:
#Submission
test_df = pd.read_csv('../input/enerjisa-enerji-veri-maratonu/sample_submission.csv')

y_sub = np.mean(test_preds, axis=0)
y_sub = pd.DataFrame(y_sub)
y_sub['DateTime'] = np.unique(pd.to_datetime(test_df.DateTime).dt.date)
y_sub[[0,1,2,3,4,5,6,18,19,20,21,22,23]] = 0
y_sub = y_sub.melt(id_vars=['DateTime']).sort_values(by=['DateTime', 'variable'])
y_sub.value = y_sub.value.apply(lambda x: 0 if x<0 else x)
y_sub.DateTime = test_df.DateTime.values
y_sub.drop('variable', axis=1, inplace=True)
y_sub.columns = ['DateTime', 'Generation']
y_sub.to_csv('submission.csv', index=False)